## Parte 1 - Importando Bibliotecas

In [76]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
import win32com.client as win32

from meu_email import email

import os
import time
import pandas as pd

chrome_options = Options()

# Usando um user-agent de um navegador real (ajuda a evitar detecção)
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
# Configuração para desabilitar o erro de "Esse navegador ou app pode não ser seguro"
chrome_options.add_argument("--disable-blink-features=AutomationControlled")

# Força salvar na pasta do Anaconda
os.environ['WDM_CACHE_DIR'] = r'C:\Users\netin\anaconda3\webdrivers'

buscas = pd.read_excel('buscas.xlsx')
display(buscas)

,Nome,Termos banidos,Preço mínimo,Preço máximo
0,iphone 12 64gb,mini watch,1500,3500
1,rtx 3060,zota galax,2000,4500


## Parte 2 - Parametrizações

In [71]:
#parametros
lista = buscas['Nome']
banidos = buscas['Termos banidos']
preco_minimo = buscas['Preço mínimo']
preco_maximo = buscas['Preço máximo']
dicionario = {produto: [] for produto in lista}
display(dicionario)

{'iphone 12 64gb': [], 'rtx 3060': []}

## Parte 3 - Google Shopping

In [83]:
def busca_google_shopping(dicionario):
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    driver.get("https://www.google.com")
    time.sleep(2)

    for i in range(len(lista)):
        termo = lista[i]
        barra = driver.find_element(By.NAME, "q")
        barra.clear()
        barra.send_keys(termo)
        barra.send_keys(Keys.RETURN)
        time.sleep(5)

        try:
            shopping = driver.find_element(By.XPATH, '//a//div[text()="Shopping"]')
            shopping.click()
            time.sleep(3)

            produtos = driver.find_elements(By.CSS_SELECTOR, 'div.sh-dgr__grid-result')

            for produto in produtos:
                try:
                    nome = produto.find_element(By.CSS_SELECTOR, 'h3').text.lower()
                    if any(banido.lower() in nome for banido in banidos[i].split(',')):
                        continue

                    preco_txt = produto.find_element(By.CSS_SELECTOR, '.a8Pemb').text
                    preco = float(preco_txt.replace('R$', '').replace('.', '').replace(',', '.').strip())

                    link = produto.find_element(By.CSS_SELECTOR, 'a.shntl').get_attribute('href')

                    if preco_minimo[i] <= preco <= preco_maximo[i]:
                        for chave in dicionario:
                            if chave.lower() in nome:
                                dicionario[chave].append((nome, preco, link))
                                break
                except Exception as e:
                    continue
        except:
            continue

    driver.quit()
    return dicionario


## Parte 4 - Criação da Tabela


In [84]:
dicionario = busca_google_shopping(dicionario)

tabela = pd.DataFrame(
    [(chave, nome, preco, link) for chave, valores in dicionario.items() for nome, preco, link in valores],
    columns=['categoria', 'produto', 'preco', 'link']
)

tabela = tabela.sort_values(by='preco')
display(tabela)

tabela.to_excel('resultado.xlsx', index=False)
if os.path.exists('resultado.xlsx'):
    print("Arquivo Excel criado com sucesso!")
else:
    print("Erro ao criar o arquivo Excel.")

,categoria,produto,preco,link
43,iphone 12 64gb,apple iphone 12 64gb verde,1600.00,https://www.google.com/url?url=https://sp.olx....
17,iphone 12 64gb,apple iphone 12 64gb verde,1600.00,https://www.google.com/url?url=https://sp.olx....
26,iphone 12 64gb,apple iphone 12 64gb azul - outlet,1679.92,https://www.google.com/url?url=https://www.tro...
23,iphone 12 64gb,iphone 12 64gb / oferta especial,1799.00,https://www.google.com/url?url=https://pr.olx....
38,iphone 12 64gb,iphone 12 64gb,1800.00,https://www.google.com/url?url=https://sp.olx....
...,...,...,...,...
51,rtx 3060,placa de vídeo geforce rtx 3060 gaming oc 12gb...,4320.00,https://www.google.com/url?url=https://br.octo...
66,rtx 3060,"pc gamer ludic by bluepc - amd ryzen 5 5500, g...",4329.00,https://www.google.com/url?url=https://www.kab...
89,rtx 3060,"pc gamer ludic by bluepc - amd ryzen 5 5500, g...",4329.00,https://www.google.com/url?url=https://www.kab...
57,rtx 3060,"pc gamer completo 3green hunter, intel core i7...",4487.99,https://www.google.com/url?url=https://www.ame...


Arquivo Excel criado com sucesso!


# Parte 5 - Enviando email

In [85]:
outlook = win32.Dispatch('outlook.application')
mail = outlook.CreateItem(0)

mail.To = email
mail.Subject = 'Produto(s) Encontrado(s) na faixa de preço desejada'
mail.HTMLBody = f"""
<p>Prezados,</p>
<p>Encontramos alguns produtos em oferta dentro da faixa de preço desejada. Segue tabela com detalhes:</p>
{tabela.to_html(index=False, escape=False)}
<p>Qualquer dúvida, estou à disposição.</p>
<p>Att.,</p>
"""
try:
    mail.Send()
    print("Email enviado com sucesso!")
except Exception as e:
    print(f"Erro ao enviar o e-mail: {e}")

Email enviado com sucesso!
